# **Quantum Computing — the Practical Way**
### *QPlayLearn*

## **Installation**

First, we install the packages we need in the current environment. They will not be installed on your local machine.

In [ ]:
# Qiskit is the open-source library for quantum computing founded by IBM
! pip install qiskit qiskit-aer qiskit-ibm-runtime 
! pip install matplotlib pylatexenc

## **Importing packages**


We import all the packages we are going to need to run the code. 
<br> N.B. Remember to run this cell before every Sandbox!

In [ ]:
import qiskit as qk
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram

# Packages for graphical representations and plots
import matplotlib as mpl
import matplotlib.pyplot as plt

# Math library
import numpy as np

## **SANDBOX — Superdense coding**

In the previous Sandbox, you created Bell states and saw how their measurement outcomes are correlated. Now we will use a shared Bell state as a resource for communication.

The goal of superdense coding is to communicate two classical bits by sending one qubit. This does not mean that a qubit simply stores two classical bits. The protocol works because Alice and Bob already share an entangled pair, which must be prepared and distributed in advance.

We will build the protocol one step at a time.

### **1 - Alice and Bob share a Bell state**

Alice holds qubit $q_0$ and Bob holds qubit $q_1$. We begin by preparing

$$|\Phi^+\rangle = \frac{|00\rangle+|11\rangle}{\sqrt{2}}.$$

You used this same combination of gates in the previous Sandbox on Bell states.

In [ ]:
# Create a two-qubit circuit with two classical bits and two qubits
num_qubits = 2
num_bits = 2
qc = qk.QuantumCircuit(num_qubits, num_bits)

# Your turn: prepare the Bell state |Phi+>


qc.draw(output="mpl")

### **2 — Alice chooses a two-bit message**

Alice wants to send one of four possible messages: `00`, `01`, `10` or `11`. She encodes it by applying gates only to her qubit, $q_0$.

| Message | Alice’s operation | Shared state |
|:---:|:---:|:---:|
| `00` | $I$ — no change | $\|\Phi^+\rangle$ |
| `01` | $X$ | $\|\Psi^+\rangle$ |
| `10` | $Z$ | $\|\Phi^-\rangle$ |
| `11` | $XZ$ | $-\|\Psi^-\rangle$ |

Each operation transforms $|\Phi^+\rangle$ into a different Bell state. The minus sign in the last row is only a global phase, so it does not affect any measurement outcome.

<details>
<summary><strong>Where does the encoding table come from?</strong></summary>

Alice and Bob begin with

$$|\Phi^+\rangle=\frac{1}{\sqrt{2}}(|00\rangle+|11\rangle),$$

where the first qubit belongs to Alice. Applying $X$ exchanges $|0\rangle$ and $|1\rangle$, while applying $Z$ changes the sign of the $|1\rangle$ component:

$$\begin{aligned}
I|\Phi^+\rangle &= \frac{|00\rangle+|11\rangle}{\sqrt{2}}=|\Phi^+\rangle,\\
X|\Phi^+\rangle &= \frac{|10\rangle+|01\rangle}{\sqrt{2}}=|\Psi^+\rangle,\\
Z|\Phi^+\rangle &= \frac{|00\rangle-|11\rangle}{\sqrt{2}}=|\Phi^-\rangle,\\
XZ|\Phi^+\rangle &= \frac{|10\rangle-|01\rangle}{\sqrt{2}}=-|\Psi^-\rangle.
\end{aligned}$$

This is how Alice’s two-bit message selects one of four distinguishable Bell states.

</details>

Try changing `message` and run the circuit for each of the four possible messages. Does Bob recover all of them?

In [ ]:
# Choose the two classical bits Alice wants to send
message = "10"

if message not in {"00", "01", "10", "11"}:
    raise ValueError("The message must be 00, 01, 10 or 11.")

# The first bit controls Z; the second bit controls X
if message[0] == "1":
    qc.z(0)
if message[1] == "1":
    qc.x(0)

qc.draw(output="mpl")

### **3 — Alice sends her qubit to Bob**

After encoding the message, Alice sends $q_0$ to Bob. This is the only qubit transmitted during the communication stage. Bob now holds both qubits and can decode the message.

In our circuit diagram, nothing happens mathematically at this step: the transfer changes who holds the qubit, not its quantum state.

### **4 — Bob decodes the message**

To recover the message, Bob needs to determine which Bell state the pair is in. However, measuring the qubits directly in the computational basis would not distinguish all four Bell states. He must first transform them into the corresponding computational-basis states.

To do this, Bob reverses the operations used to create the Bell pair: first a CNOT, then a Hadamard gate. This maps the four Bell states onto the four computational-basis states, which Bob can measure to recover Alice’s two-bit message.

In [ ]:
# Bob decodes the Bell state
qc.cx(0, 1)
qc.h(0)

qc.draw(output="mpl")

### **5 — Bob measures the qubits**

Bob measures both qubits to recover Alice’s message. We connect $q_0$ to the left displayed classical bit and $q_1$ to the right one, so Qiskit prints the result in the same order in which Alice wrote the message.

In [ ]:
# Map q0 to c1 and q1 to c0 so the displayed bitstring matches the message
qc.measure(0, 1)
qc.measure(1, 0)

qc.draw(output="mpl")

Run the circuit on a simulator, and try increasing or decreasing the number of shots. Why does the same bitstring appear every time on the ideal simulator?

In [ ]:
# Run the circuit on a simulator
simulator = AerSimulator()
result = simulator.run(qc, shots=100).result()
counts = result.get_counts()

print("Message sent:    ", message)
print("Message received:", max(counts, key=counts.get))
plot_histogram(counts)

If everything worked, Bob’s measurement reproduces Alice’s two-bit message. Unlike the probabilistic outcomes you observed when measuring a Bell state directly, the decoded result is deterministic on an ideal simulator.

It was not for free! Alice manipulated and sent one qubit, but the protocol also consumed an entangled pair that Alice and Bob had shared beforehand.